In [ ]:
# Install dependencies (run once).
!pip install tensorflow pandas numpy scikit-learn matplotlib scipy

# Adaptive CPU Auto-Scaling - Hybrid LSTM+MLP with Continual Learning (V3)

A clean, readable notebook. It is self-contained (no project imports) and runs with only the dataset
`alibaba_timeseries_full.csv`.

**Idea.** Forecast CPU demand 30 minutes ahead with a two-branch network (an LSTM for the time series
and an MLP for the static job features). Train it over 4 time-ordered chunks and use two continual
learning tricks so it does not forget older patterns: **EWC** protects the weights that mattered for
past chunks, and **Experience Replay** mixes a few old samples into each new chunk.

**How to use.** Run Part A once (it just defines functions), then run Part B top to bottom. Part B is the
actual pipeline, one step per cell, each prints its output.

---

# Part A - Definitions


## Imports


In [ ]:
import os, json, math, copy, pickle
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, BatchNormalization, concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.utils import register_keras_serializable

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error

## Settings

All knobs in one small place. The validation studies copy this and tweak single fields.


In [ ]:
@dataclass
class Config:
    data_path: str = "alibaba_timeseries_full.csv"
    history_length: int = 24      # look-back window (24 x 5 min = 2 hours)
    forecast_steps: int = 6       # predict 30 minutes ahead
    n_chunks: int = 4             # continual-learning chunks
    epochs: int = 30
    batch_size: int = 32
    ewc_lambda: float = 100.0     # EWC penalty strength
    replay_memory: int = 2000     # replay buffer size
    replay_ratio: float = 0.2     # fraction of replay samples per batch
    roll_window: int = 12         # rolling-stats window (1 hour)
    sla_tol: float = 0.15         # SLA tolerance band
    seed: int = 42


# Columns the model reads. app_id is appended to the static features inside prepare_data.
TEMPORAL_FEATURES = [
    "cpu_demand", "cpu_diff", "cpu_roll_mean", "cpu_roll_std", "cpu_roll_min",
    "cpu_roll_max", "hour_sin", "hour_cos", "dow_sin", "dow_cos",
]
STATIC_FEATURES = [
    "gpu_request_mean", "memory_request_mean", "rdma_request_mean",
    "role_hn_fraction", "instance_count", "max_instance_per_node",
]
TARGET = "cpu_demand"


def set_seeds(seed):
    # Fix all RNGs and keep things on CPU / float32 for reproducibility.
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
    try:
        tf.config.set_visible_devices([], "GPU")
    except Exception:
        pass
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.keras.mixed_precision.set_global_policy("float32")

## Data loading and feature engineering


In [ ]:
def load_data(path):
    df = pd.read_csv(path)
    print(f"rows: {len(df):,} | apps: {df['app_name'].nunique()}")
    return df


def engineer_features(df, roll_window):
    # Add diff, rolling stats and cyclic time features, per app (so rolling stays app-specific).
    out = []
    for app, g in df.groupby("app_name", sort=False):
        g = g.sort_values("timestamp").copy()
        g["cpu_diff"] = g["cpu_demand"].diff().fillna(0)
        g["cpu_roll_mean"] = g["cpu_demand"].rolling(roll_window, min_periods=1).mean()
        g["cpu_roll_std"] = g["cpu_demand"].rolling(roll_window, min_periods=1).std().fillna(0)
        g["cpu_roll_min"] = g["cpu_demand"].rolling(roll_window, min_periods=1).min()
        g["cpu_roll_max"] = g["cpu_demand"].rolling(roll_window, min_periods=1).max()
        day, week = 86400, 604800
        g["hour_sin"] = np.sin(2 * np.pi * g["timestamp"] / day)
        g["hour_cos"] = np.cos(2 * np.pi * g["timestamp"] / day)
        g["dow_sin"] = np.sin(2 * np.pi * g["timestamp"] / week)
        g["dow_cos"] = np.cos(2 * np.pi * g["timestamp"] / week)
        out.append(g)
    return pd.concat(out, ignore_index=True)

## Sequences, scaling and chunk split


In [ ]:
def make_sequences(temporal, static, target, history, horizon):
    # Slide a window over one app: X_seq = history window, X_static = row at window end,
    # y = target horizon steps later.
    X_seq, X_static, y = [], [], []
    for i in range(len(temporal) - history - horizon + 1):
        X_seq.append(temporal[i:i + history])
        X_static.append(static[i + history])
        y.append(target[i + history + horizon - 1])
    return np.array(X_seq), np.array(X_static), np.array(y)


def prepare_data(df, cfg, temporal_features, static_features, target, out_dir="."):
    df = df.copy()
    # Give each app a numeric id and add it to the static features.
    app_ids = {a: i for i, a in enumerate(df["app_name"].unique())}
    df["app_id"] = df["app_name"].map(app_ids)
    static_full = static_features + ["app_id"]

    # Build sequences app by app, then pool them.
    seqs, statics, ys = [], [], []
    min_len = cfg.history_length + cfg.forecast_steps + 10
    for _, g in df.groupby("app_name", sort=False):
        g = g.sort_values("timestamp")
        if len(g) < min_len:
            continue
        xs, st, y = make_sequences(
            g[temporal_features].values.astype("float32"),
            g[static_full].values.astype("float32"),
            g[target].values.astype("float32"),
            cfg.history_length, cfg.forecast_steps,
        )
        seqs.append(xs); statics.append(st); ys.append(y)
    X_seq = np.concatenate(seqs)
    X_static = np.concatenate(statics)
    y = np.concatenate(ys)

    # Fit and apply scalers, then save them for inference.
    n_temporal, n_static = X_seq.shape[2], X_static.shape[1]
    t_scaler, s_scaler, y_scaler = StandardScaler(), StandardScaler(), StandardScaler()
    X_seq = t_scaler.fit_transform(X_seq.reshape(-1, n_temporal)).reshape(X_seq.shape)
    X_static = s_scaler.fit_transform(X_static)
    y = y_scaler.fit_transform(y.reshape(-1, 1)).flatten()
    out = Path(out_dir)
    for name, sc in [("temporal_scaler", t_scaler), ("static_scaler", s_scaler), ("target_scaler", y_scaler)]:
        with open(out / f"{name}.pkl", "wb") as f:
            pickle.dump(sc, f)

    # Split into sequential chunks (no shuffle) to mimic an evolving workload.
    size = len(y) // cfg.n_chunks
    chunks = []
    for i in range(cfg.n_chunks):
        a = i * size
        b = (i + 1) * size if i < cfg.n_chunks - 1 else len(y)
        chunks.append({"X_seq": X_seq[a:b], "X_static": X_static[a:b], "y": y[a:b]})

    print(f"sequences: {len(y):,} | temporal: {n_temporal} | static: {n_static}")
    print("chunk sizes:", [len(c["y"]) for c in chunks])
    return chunks, t_scaler, s_scaler, y_scaler, n_temporal, n_static

## Models

The hybrid model, plus single-branch versions used only by the ablation study.


In [ ]:
def build_hybrid_model(history, n_temporal, n_static):
    # LSTM branch for the time series.
    seq_in = Input(shape=(history, n_temporal), name="temporal_input")
    x = LSTM(128, return_sequences=True, dropout=0.2)(seq_in)
    x = LSTM(64, return_sequences=True, dropout=0.2)(x)
    x = LSTM(32)(x)
    seq_embed = Dense(16, activation="relu")(x)
    # MLP branch for the static features.
    stat_in = Input(shape=(n_static,), name="static_input")
    s = Dense(64, activation="relu")(stat_in)
    s = BatchNormalization()(s)
    s = Dropout(0.2)(s)
    s = Dense(32, activation="relu")(s)
    stat_embed = Dense(16, activation="relu")(s)
    # Fuse and predict one value.
    f = concatenate([seq_embed, stat_embed])
    f = Dense(16, activation="relu")(f)
    out = Dense(1, activation="linear")(f)
    model = Model([seq_in, stat_in], out)
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model


def build_lstm_only(history, n_temporal):
    inp = Input(shape=(history, n_temporal))
    x = LSTM(128, return_sequences=True, dropout=0.2)(inp)
    x = LSTM(64, return_sequences=True, dropout=0.2)(x)
    x = LSTM(32)(x)
    out = Dense(1, activation="linear")(x)
    m = Model(inp, out)
    m.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return m


def build_mlp_only(n_static):
    inp = Input(shape=(n_static,))
    x = Dense(64, activation="relu")(inp)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    x = Dense(32, activation="relu")(x)
    x = Dense(16, activation="relu")(x)
    out = Dense(1, activation="linear")(x)
    m = Model(inp, out)
    m.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return m

## Elastic Weight Consolidation (EWC)

Protects the weights that were important for past chunks, using a diagonal Fisher estimate.


In [ ]:
@register_keras_serializable()
def loss_with_ewc(y_true, y_pred, model, penalty_fn):
    # Normal MSE plus the EWC penalty on the current weights.
    return MeanSquaredError()(y_true, y_pred) + penalty_fn(model.trainable_weights)


class EWC:
    def __init__(self, model, fisher_multiplier):
        self.model = model
        self.fisher_multiplier = fisher_multiplier
        self.fisher = None       # importance of each weight
        self.old_params = None   # weights to anchor to

    def compute_fisher(self, x_seq, x_static, y, sample_size=200):
        # Estimate weight importance as squared gradients on a sample of the previous chunk.
        n = len(y)
        if n > sample_size:
            idx = np.random.choice(n, sample_size, replace=False)
            x_seq, x_static, y = x_seq[idx], x_static[idx], y[idx]
        self.old_params = [w.numpy().copy() for w in self.model.trainable_weights]
        with tf.GradientTape() as tape:
            preds = self.model([x_seq, x_static], training=False)
            loss = tf.reduce_mean(tf.square(tf.cast(y.reshape(-1, 1), tf.float32) - preds))
        grads = tape.gradient(loss, self.model.trainable_weights)
        self.fisher = [tf.square(g) if g is not None else tf.zeros_like(w)
                       for g, w in zip(grads, self.model.trainable_weights)]

    def penalty(self, weights):
        # Quadratic pull back toward the old weights, scaled by importance.
        if self.fisher is None:
            return tf.constant(0.0)
        total = tf.constant(0.0)
        for f, old, cur in zip(self.fisher, self.old_params, weights):
            total += tf.reduce_sum(f * tf.square(cur - old))
        return 0.5 * self.fisher_multiplier * total

    def train(self, x_seq, x_static, y, epochs, batch_size, validation_data=None):
        # Compile with MSE + penalty, fit, then restore plain MSE for clean prediction.
        def loss_fn(y_true, y_pred):
            return loss_with_ewc(y_true, y_pred, self.model, self.penalty)
        self.model.compile(optimizer="adam", loss=loss_fn, metrics=["mae"])
        hist = self.model.fit([x_seq, x_static], y, epochs=epochs, batch_size=batch_size,
                              validation_data=validation_data, verbose=1)
        self.model.compile(optimizer="adam", loss="mse", metrics=["mae"])
        return hist

## Experience Replay (ER)

Keeps a buffer of old samples and mixes a fraction into each new chunk.


In [ ]:
class ExperienceReplay:
    def __init__(self, memory_size, replay_ratio):
        self.memory_size = memory_size
        self.replay_ratio = replay_ratio
        self.seqs, self.statics, self.ys = [], [], []

    @property
    def size(self):
        return len(self.ys)

    def update(self, x_seq, x_static, y):
        # Add this chunk's samples, then randomly trim back to the buffer size.
        self.seqs.extend(list(x_seq))
        self.statics.extend(list(x_static))
        self.ys.extend(float(v) for v in y)
        if self.size > self.memory_size:
            keep = np.random.choice(self.size, self.memory_size, replace=False)
            self.seqs = [self.seqs[i] for i in keep]
            self.statics = [self.statics[i] for i in keep]
            self.ys = [self.ys[i] for i in keep]

    def mix(self, x_seq, x_static, y):
        # Append a replay_ratio share of old samples to the new batch.
        n = min(int(len(y) * self.replay_ratio), self.size)
        if n == 0:
            return x_seq, x_static, y
        idx = np.random.choice(self.size, n, replace=False)
        rs = np.array([self.seqs[i] for i in idx])
        rt = np.array([self.statics[i] for i in idx])
        ry = np.array([self.ys[i] for i in idx])
        return (np.concatenate([x_seq, rs]),
                np.concatenate([x_static, rt]),
                np.concatenate([y.flatten(), ry.flatten()]))

## Metrics


In [ ]:
def eval_real(model, x_seq, x_static, y_scaled, y_scaler):
    # Predict, then invert scaling so errors are in real CPU units.
    inp = [x_seq, x_static] if x_static is not None else x_seq
    pred = y_scaler.inverse_transform(model.predict(inp, verbose=0).reshape(-1, 1)).flatten()
    real = y_scaler.inverse_transform(np.asarray(y_scaled).reshape(-1, 1)).flatten()
    mae = float(mean_absolute_error(real, pred))
    rmse = float(np.sqrt(mean_squared_error(real, pred)))
    mape = float(np.mean(np.abs((real - pred) / (real + 1e-8))) * 100)
    return mae, rmse, mape


def provisioning_metrics(y_true, y_pred, sla_tol=0.15):
    # Over/under provisioning as a share of total demand, plus SLA violations.
    diff = y_pred - y_true
    total = float(np.sum(np.abs(y_true))) + 1e-8
    over = float(np.sum(np.maximum(diff, 0)) / total * 100)
    under = float(np.sum(np.maximum(-diff, 0)) / total * 100)
    sla = float(np.mean(y_pred < (1 - sla_tol) * y_true) * 100)
    return over, under, sla


def bwt_scalar(bwt_mat, chunk_results):
    # Backward transfer: mean of (MAE on chunk j after the last chunk) - (MAE when j was new).
    # Negative means the model improved on old chunks (good).
    n = len(chunk_results)
    scores = []
    for j in range(n - 1):
        last = bwt_mat.get(n - 1, {}).get(j)
        if last is not None:
            scores.append(last - chunk_results[j]["mae"])
    return float(np.mean(scores)) if scores else float("nan")

## Continual training loop (the proposed model)


In [ ]:
def train_proposed(cfg, chunks, n_temporal, n_static, y_scaler):
    model = build_hybrid_model(cfg.history_length, n_temporal, n_static)
    ewc = EWC(model, cfg.ewc_lambda)
    replay = ExperienceReplay(cfg.replay_memory, cfg.replay_ratio)
    chunk_results, bwt_mat, histories, val_sets = [], {}, [], []
    prev = None

    for i, chunk in enumerate(chunks):
        # Time-ordered 80/20 split.
        split = int(0.8 * len(chunk["y"]))
        xs_tr, xs_val = chunk["X_seq"][:split], chunk["X_seq"][split:]
        st_tr, st_val = chunk["X_static"][:split], chunk["X_static"][split:]
        y_tr, y_val = chunk["y"][:split], chunk["y"][split:]
        val_sets.append((xs_val, st_val, y_val))

        # From chunk 2 on, measure weight importance on the previous chunk first.
        if i > 0 and prev is not None:
            ewc.compute_fisher(*prev)

        # Add replay samples, then train (plain MSE for chunk 1, MSE+EWC after).
        xs_mix, st_mix, y_mix = replay.mix(xs_tr, st_tr, y_tr)
        print(f"chunk {i + 1}/{cfg.n_chunks}: train={len(y_mix)} (incl. replay)")
        if i == 0:
            hist = model.fit([xs_mix, st_mix], y_mix, epochs=cfg.epochs, batch_size=cfg.batch_size,
                             validation_data=([xs_val, st_val], y_val), verbose=1)
        else:
            hist = ewc.train(xs_mix, st_mix, y_mix, cfg.epochs, cfg.batch_size,
                             validation_data=([xs_val, st_val], y_val))
        histories.append(hist)

        # Score this chunk, and re-score all earlier chunks (backward transfer).
        mae, rmse, mape = eval_real(model, xs_val, st_val, y_val, y_scaler)
        chunk_results.append({"chunk": i + 1, "mae": mae, "rmse": rmse, "mape": mape})
        print(f"  chunk {i + 1}: MAE={mae:.4f} RMSE={rmse:.4f}")
        bwt_mat[i] = {j: eval_real(model, pxs, pst, py, y_scaler)[0]
                      for j, (pxs, pst, py) in enumerate(val_sets[:-1])}

        # Remember this chunk for replay and for the next Fisher step.
        replay.update(xs_tr, st_tr, y_tr)
        prev = (xs_tr.copy(), st_tr.copy(), y_tr.copy())

    # Final-chunk predictions in real units, for the baseline comparison.
    xs_f, st_f, y_f = val_sets[-1]
    y_pred = y_scaler.inverse_transform(model.predict([xs_f, st_f], verbose=0).reshape(-1, 1)).flatten()
    y_real = y_scaler.inverse_transform(y_f.reshape(-1, 1)).flatten()
    return model, chunk_results, bwt_mat, histories, val_sets, y_real, y_pred

## Baselines

Four references, all scored on the same final-chunk validation set.


In [ ]:
def run_baselines(cfg, chunks, n_temporal, n_static, xs_final, st_final, y_real, y_pred, y_scaler):
    c1 = chunks[0]
    sp = int(0.8 * len(c1["y"]))
    xs_tr, st_tr, y_tr = c1["X_seq"][:sp], c1["X_static"][:sp], c1["y"][:sp]

    # HPA-reactive: just shift the actual series by the forecast horizon.
    hpa = np.roll(y_real, cfg.forecast_steps)
    hpa[:cfg.forecast_steps] = y_real[:cfg.forecast_steps]

    # Static LSTM and static hybrid: trained once on chunk 1.
    m_lstm = build_lstm_only(cfg.history_length, n_temporal)
    m_lstm.fit(xs_tr, y_tr, epochs=cfg.epochs, batch_size=cfg.batch_size, verbose=0)
    p_lstm = y_scaler.inverse_transform(m_lstm.predict(xs_final, verbose=0).reshape(-1, 1)).flatten()

    m_hyb = build_hybrid_model(cfg.history_length, n_temporal, n_static)
    m_hyb.fit([xs_tr, st_tr], y_tr, epochs=cfg.epochs, batch_size=cfg.batch_size, verbose=0)
    p_hyb = y_scaler.inverse_transform(m_hyb.predict([xs_final, st_final], verbose=0).reshape(-1, 1)).flatten()

    # Periodic retrain: refit on every chunk in turn.
    m_per = build_hybrid_model(cfg.history_length, n_temporal, n_static)
    for c in chunks:
        s = int(0.8 * len(c["y"]))
        m_per.fit([c["X_seq"][:s], c["X_static"][:s]], c["y"][:s],
                  epochs=cfg.epochs, batch_size=cfg.batch_size, verbose=0)
    p_per = y_scaler.inverse_transform(m_per.predict([xs_final, st_final], verbose=0).reshape(-1, 1)).flatten()

    def row(name, yt, yp):
        # Filtered MAPE ignores near-zero demand (below the 10th percentile).
        nz = yt > 0
        p10 = float(np.percentile(yt[nz], 10)) if nz.any() else 0.0
        m = yt > p10
        ref = m if m.any() else np.ones_like(yt, dtype=bool)
        mape = float(np.mean(np.abs((yt[ref] - yp[ref]) / (yt[ref] + 1e-8))) * 100)
        over, under, sla = provisioning_metrics(yt, yp)
        return {"Method": name, "MAE": float(mean_absolute_error(yt, yp)),
                "RMSE": float(np.sqrt(mean_squared_error(yt, yp))),
                "MAPE% (filtered)": mape, "OverProv%": over, "UnderProv%": under, "SLA_Viol%": sla}

    preds = {"HPA-Reactive": hpa, "Static LSTM": p_lstm, "Static Hybrid": p_hyb,
             "Periodic Retrain": p_per, "Proposed (EWC+ER)": y_pred}
    table = pd.DataFrame([row(k, y_real, v) for k, v in preds.items()]).set_index("Method")
    return table, preds

## Save tables and plot


In [ ]:
def save_tables(out_dir, chunk_results, results_df):
    out = Path(out_dir)
    pd.DataFrame(chunk_results).to_csv(out / "chunk_metrics.csv", index=False)
    results_df.to_csv(out / "baseline_results.csv")
    with open(out / "chunk_metrics.json", "w") as f:
        json.dump(chunk_results, f, indent=2)
    with open(out / "baseline_results.json", "w") as f:
        json.dump(results_df.reset_index().to_dict(orient="records"), f, indent=2)


def plot_results(out_dir, chunk_results, bwt_mat, y_real, preds, results_df):
    fig, ax = plt.subplots(2, 2, figsize=(14, 9))
    labels = [f"C{r['chunk']}" for r in chunk_results]
    ax[0, 0].plot(labels, [r["mae"] for r in chunk_results], "o-", label="MAE")
    ax[0, 0].plot(labels, [r["rmse"] for r in chunk_results], "s-", label="RMSE")
    ax[0, 0].set_title("Error across chunks"); ax[0, 0].legend(); ax[0, 0].grid(alpha=0.3)

    for after in sorted(bwt_mat):
        ks = sorted(bwt_mat[after])
        if ks:
            ax[0, 1].plot([f"C{k+1}" for k in ks], [bwt_mat[after][k] for k in ks], "D-", label=f"after C{after+1}")
    ax[0, 1].set_title("Backward transfer (MAE on old chunks)"); ax[0, 1].legend(fontsize=8); ax[0, 1].grid(alpha=0.3)

    n = min(200, len(y_real))
    ax[1, 0].plot(y_real[:n], "k", label="actual")
    ax[1, 0].plot(preds["Proposed (EWC+ER)"][:n], label="proposed")
    ax[1, 0].plot(preds["Periodic Retrain"][:n], "--", label="periodic")
    ax[1, 0].set_title("Predicted vs actual (final chunk)"); ax[1, 0].legend(fontsize=8); ax[1, 0].grid(alpha=0.3)

    ax[1, 1].bar(results_df.index, results_df["MAE"])
    ax[1, 1].set_title("Final-chunk MAE by method")
    ax[1, 1].tick_params(axis="x", rotation=20, labelsize=8); ax[1, 1].grid(axis="y", alpha=0.3)

    fig.tight_layout()
    fig.savefig(Path(out_dir) / "evaluation_results.png", dpi=150, bbox_inches="tight")
    plt.show()

## Validation studies

Ablation, multi-seed, naive fine-tuning, sensitivity and cross-app. Each reuses the functions above.


In [ ]:
def ablation_variant(cfg, chunks, n_temporal, n_static, y_scaler, mode):
    c = copy.copy(cfg)
    if mode == "hybrid_ewc":
        c.replay_ratio = 0.0           # EWC only
    if mode == "hybrid_er":
        c.ewc_lambda = 0.0             # ER only
    if mode in ("hybrid_ewc", "hybrid_er"):
        _, _, _, _, _, yr, yp = train_proposed(c, chunks, n_temporal, n_static, y_scaler)
        return float(mean_absolute_error(yr, yp)), float(np.sqrt(mean_squared_error(yr, yp)))

    # Single-branch / no-continual-learning variants: plain fit over every chunk.
    if mode == "lstm_only":
        model, pick = build_lstm_only(cfg.history_length, n_temporal), (lambda xs, st: xs)
    elif mode == "mlp_only":
        model, pick = build_mlp_only(n_static), (lambda xs, st: st)
    else:  # hybrid_no_cl
        model, pick = build_hybrid_model(cfg.history_length, n_temporal, n_static), (lambda xs, st: [xs, st])

    val = []
    for ch in chunks:
        s = int(0.8 * len(ch["y"]))
        val.append((ch["X_seq"][s:], ch["X_static"][s:], ch["y"][s:]))
        model.fit(pick(ch["X_seq"][:s], ch["X_static"][:s]), ch["y"][:s],
                  epochs=cfg.epochs, batch_size=cfg.batch_size, verbose=0)
    xs, st, y = val[-1]
    yp = y_scaler.inverse_transform(model.predict(pick(xs, st), verbose=0).reshape(-1, 1)).flatten()
    yr = y_scaler.inverse_transform(y.reshape(-1, 1)).flatten()
    return float(mean_absolute_error(yr, yp)), float(np.sqrt(mean_squared_error(yr, yp)))


def run_ablation(cfg, chunks, n_temporal, n_static, y_scaler, proposed_mae, proposed_rmse):
    variants = [("A", "LSTM-only (no MLP)", "lstm_only"), ("B", "MLP-only (no LSTM)", "mlp_only"),
                ("C", "Hybrid (no CL)", "hybrid_no_cl"), ("D", "Hybrid + EWC only", "hybrid_ewc"),
                ("E", "Hybrid + ER only", "hybrid_er")]
    rows = []
    for lab, desc, mode in variants:
        mae, rmse = ablation_variant(cfg, chunks, n_temporal, n_static, y_scaler, mode)
        print(f"  {lab} {desc}: MAE={mae:.4f}")
        rows.append({"Variant": lab, "Description": desc, "MAE": mae, "RMSE": rmse})
    rows.append({"Variant": "F", "Description": "Proposed (EWC+ER)", "MAE": proposed_mae, "RMSE": proposed_rmse})
    return pd.DataFrame(rows)


def run_multi_seed(cfg, chunks, n_temporal, n_static, y_scaler, out_dir="."):
    rows = []
    for seed in [42, 43, 44, 45, 46]:
        c = copy.copy(cfg); c.seed = seed
        set_seeds(seed)
        _, _, _, _, _, yr, yp = train_proposed(c, chunks, n_temporal, n_static, y_scaler)
        mae = float(mean_absolute_error(yr, yp)); print(f"  seed {seed}: MAE={mae:.4f}")
        rows.append({"seed": seed, "mae": mae})
    df = pd.DataFrame(rows); vals = df["mae"].values
    mean, std = float(np.mean(vals)), float(np.std(vals, ddof=1))
    lo, hi = stats.t.interval(0.95, len(vals) - 1, loc=mean, scale=std / math.sqrt(len(vals)))
    summary = pd.DataFrame([{"seed": "summary", "mae": mean, "std": std, "ci_lo_95": float(lo), "ci_hi_95": float(hi)}])

    # Significance vs each baseline (needs baseline_results.json from the main run).
    p = Path(out_dir) / "baseline_results.json"
    if p.exists():
        sig = []
        for rec in json.load(open(p)):
            if rec["Method"] == "Proposed (EWC+ER)":
                continue
            bmae = float(rec["MAE"]); t, pv = stats.ttest_1samp(vals, bmae)
            d = (mean - bmae) / std if std > 0 else float("nan")
            sig.append({"comparison": f"Proposed vs {rec['Method']}", "baseline_mae": round(bmae, 4),
                        "p_value": round(float(pv), 4), "cohens_d": round(float(d), 4)})
        if sig:
            pd.DataFrame(sig).to_csv(Path(out_dir) / "significance_tests.csv", index=False)
    return pd.concat([df, summary], ignore_index=True)


def run_naive_ft(cfg, chunks, n_temporal, n_static, y_scaler):
    # Sequential fine-tuning with no EWC and no ER: the forgetting reference.
    model = build_hybrid_model(cfg.history_length, n_temporal, n_static)
    diag, val = [], []
    for ch in chunks:
        s = int(0.8 * len(ch["y"]))
        val.append((ch["X_seq"][s:], ch["X_static"][s:], ch["y"][s:]))
        model.compile(optimizer="adam", loss="mse", metrics=["mae"])
        model.fit([ch["X_seq"][:s], ch["X_static"][:s]], ch["y"][:s],
                  epochs=cfg.epochs, batch_size=cfg.batch_size, verbose=0)
        diag.append(eval_real(model, *val[-1], y_scaler)[0])
    final = [eval_real(model, xs, st, y, y_scaler)[0] for xs, st, y in val[:-1]]
    scores = [final[j] - diag[j] for j in range(len(final))]
    return {"naive_ft_bwt": float(np.mean(scores)) if scores else float("nan"),
            "diagonal_maes": diag, "final_maes": final}


def run_sensitivity(cfg, chunks, n_temporal, n_static, y_scaler):
    epochs = min(cfg.epochs, 10)
    res = {"lambda": [], "buffer": [], "ratio": []}
    def mae_for(c):
        _, _, _, _, _, yr, yp = train_proposed(c, chunks, n_temporal, n_static, y_scaler)
        return float(mean_absolute_error(yr, yp))
    for lam in [10.0, 100.0, 500.0, 1000.0, 5000.0]:
        c = copy.copy(cfg); c.ewc_lambda = lam; c.epochs = epochs
        res["lambda"].append({"value": lam, "mae": mae_for(c)})
    for buf in [100, 500, 1000, 2000]:
        c = copy.copy(cfg); c.replay_memory = buf; c.epochs = epochs
        res["buffer"].append({"value": buf, "mae": mae_for(c)})
    for ratio in [0.1, 0.2, 0.3, 0.4, 0.5]:
        c = copy.copy(cfg); c.replay_ratio = ratio; c.epochs = epochs
        res["ratio"].append({"value": ratio, "mae": mae_for(c)})
    return res


def run_cross_app(model_path, df, cfg, t_scaler, s_scaler, y_scaler, temporal_features, static_features, target):
    # Hold out whole apps and score the saved model on them.
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=cfg.seed)
    _, test_idx = next(gss.split(df, groups=df["app_name"]))
    hold = df.iloc[test_idx].copy()
    app_ids = {a: i for i, a in enumerate(df["app_name"].unique())}
    hold["app_id"] = hold["app_name"].map(app_ids).fillna(-1)
    static_full = static_features + ["app_id"]

    seqs, statics, ys = [], [], []
    min_len = cfg.history_length + cfg.forecast_steps + 10
    for _, g in hold.groupby("app_name", sort=False):
        g = g.sort_values("timestamp")
        if len(g) < min_len:
            continue
        xs, st, y = make_sequences(g[temporal_features].values.astype("float32"),
                                   g[static_full].values.astype("float32"),
                                   g[target].values.astype("float32"),
                                   cfg.history_length, cfg.forecast_steps)
        seqs.append(xs); statics.append(st); ys.append(y)
    if not seqs:
        return {"mae": None, "note": "no sequences"}
    X_seq, X_static, y = np.concatenate(seqs), np.concatenate(statics), np.concatenate(ys)
    nt = X_seq.shape[2]
    X_seq = t_scaler.transform(X_seq.reshape(-1, nt)).reshape(X_seq.shape)
    X_static = s_scaler.transform(X_static)

    model = tf.keras.models.load_model(str(model_path))
    yp = y_scaler.inverse_transform(model.predict([X_seq, X_static], verbose=0).reshape(-1, 1)).flatten()
    _, _, sla = provisioning_metrics(y, yp, cfg.sla_tol)
    return {"holdout_sequences": int(len(y)),
            "mae": float(mean_absolute_error(y, yp)),
            "rmse": float(np.sqrt(mean_squared_error(y, yp))),
            "sla_viol_pct": sla}

---

# Part B - Run the pipeline

Run these in order. Each step prints its result.


## Step 1 - Settings for this run


In [ ]:
cfg = Config()

# Find the dataset (current folder or one level up).
for _p in ["alibaba_timeseries_full.csv", "../alibaba_timeseries_full.csv"]:
    if Path(_p).exists():
        cfg.data_path = _p
        break

OUT_DIR = "."
RUN_VALIDATION = False   # set True to run the heavy studies in Step 11

print(cfg)

## Step 2 - Seed everything


In [ ]:
set_seeds(cfg.seed)
print("TensorFlow:", tf.__version__, "| seed:", cfg.seed)

## Step 3 - Load data


In [ ]:
df = load_data(cfg.data_path)
df.head()

## Step 4 - Feature engineering


In [ ]:
df = engineer_features(df, cfg.roll_window)
df[TEMPORAL_FEATURES].describe().round(3)

## Step 5 - Sequences, scalers and chunks


In [ ]:
chunks, t_scaler, s_scaler, y_scaler, n_temporal, n_static = prepare_data(
    df, cfg, TEMPORAL_FEATURES, STATIC_FEATURES, TARGET, OUT_DIR
)

## Step 6 - Look at the model


In [ ]:
build_hybrid_model(cfg.history_length, n_temporal, n_static).summary()

## Step 7 - Train the proposed model (slow)


In [ ]:
model, chunk_results, bwt_mat, histories, val_sets, y_real, y_pred = train_proposed(
    cfg, chunks, n_temporal, n_static, y_scaler
)
pd.DataFrame(chunk_results).round(4)

## Step 8 - Backward transfer


In [ ]:
print(f"BWT scalar: {bwt_scalar(bwt_mat, chunk_results):+.4f}  (negative = no forgetting)")
pd.DataFrame({f"after_C{a+1}": {f"eval_C{j+1}": round(v, 2) for j, v in bwt_mat[a].items()}
             for a in bwt_mat}).T

## Step 9 - Baselines


In [ ]:
xs_final, st_final = val_sets[-1][0], val_sets[-1][1]
results_df, preds = run_baselines(cfg, chunks, n_temporal, n_static, xs_final, st_final, y_real, y_pred, y_scaler)
results_df.round(4)

## Step 10 - Provisioning and SLA


In [ ]:
over, under, sla = provisioning_metrics(y_real, y_pred)
print(f"over-provisioning : {over:.2f}")
print(f"under-provisioning: {under:.2f}")
print(f"SLA violation (%) : {sla:.2f}")

## Step 11 - Save tables, model and figure


In [ ]:
save_tables(OUT_DIR, chunk_results, results_df)
model.save(Path(OUT_DIR) / "hybrid_model_ewc_er.keras")
plot_results(OUT_DIR, chunk_results, bwt_mat, y_real, preds, results_df)

## Step 12 - Optional validation studies (heavy)

These retrain models, so they only run when `RUN_VALIDATION = True` (Step 1).


In [ ]:
if RUN_VALIDATION:
    ms = run_multi_seed(cfg, chunks, n_temporal, n_static, y_scaler, OUT_DIR)
    ms.to_csv(Path(OUT_DIR) / "multi_seed_stats.csv", index=False)
    display(ms.round(4))
else:
    print("RUN_VALIDATION is False - skipping multi-seed.")

In [ ]:
if RUN_VALIDATION:
    p_mae, p_rmse = chunk_results[-1]["mae"], chunk_results[-1]["rmse"]
    abl = run_ablation(cfg, chunks, n_temporal, n_static, y_scaler, p_mae, p_rmse)
    abl.to_csv(Path(OUT_DIR) / "ablation_results.csv", index=False)
    display(abl.round(4))
else:
    print("RUN_VALIDATION is False - skipping ablation.")

In [ ]:
if RUN_VALIDATION:
    nft = run_naive_ft(cfg, chunks, n_temporal, n_static, y_scaler)
    with open(Path(OUT_DIR) / "naive_ft_results.json", "w") as f:
        json.dump(nft, f, indent=2)
    print("naive-FT BWT:", round(nft["naive_ft_bwt"], 4),
          "| proposed BWT:", round(bwt_scalar(bwt_mat, chunk_results), 4))
else:
    print("RUN_VALIDATION is False - skipping naive fine-tuning.")

In [ ]:
if RUN_VALIDATION:
    cross = run_cross_app(Path(OUT_DIR) / "hybrid_model_ewc_er.keras", df, cfg,
                          t_scaler, s_scaler, y_scaler, TEMPORAL_FEATURES, STATIC_FEATURES, TARGET)
    with open(Path(OUT_DIR) / "cross_app_results.json", "w") as f:
        json.dump(cross, f, indent=2)
    print(json.dumps(cross, indent=2))
else:
    print("RUN_VALIDATION is False - skipping cross-app.")

In [ ]:
if RUN_VALIDATION:
    sens = run_sensitivity(cfg, chunks, n_temporal, n_static, y_scaler)
    with open(Path(OUT_DIR) / "sensitivity_results.json", "w") as f:
        json.dump(sens, f, indent=2)
    print(json.dumps(sens, indent=2))
else:
    print("RUN_VALIDATION is False - skipping sensitivity.")

## Step 13 - Generated files


In [ ]:
for name in ["hybrid_model_ewc_er.keras", "temporal_scaler.pkl", "static_scaler.pkl", "target_scaler.pkl",
             "chunk_metrics.csv", "baseline_results.csv", "evaluation_results.png",
             "multi_seed_stats.csv", "ablation_results.csv", "naive_ft_results.json",
             "cross_app_results.json", "sensitivity_results.json"]:
    print(f"{name:<28} -> {'OK' if (Path(OUT_DIR) / name).exists() else 'missing'}")